# Notebook 06: Semantic Matching and Skill Gap Analysis

## Objective
Compute semantic similarity between candidate skill profiles and job descriptions.
Implement skill gap analysis using canonical token set comparison.
Produce reusable embedding artifacts for the Streamlit application.

## Methodology
1. Validate all input artifacts
2. Construct candidate skill sentences for embedding
3. Embed job descriptions from full description text
4. Embed candidate profiles from constructed skill sentences
5. Compute cosine similarity per candidate against their domain job pool
6. Implement skill gap analysis using canonical token set difference
7. Export all artifacts

## Assumptions
- Canonical tokens are the primary matching representation
- Job descriptions are embedded from full description text, not extracted skill tokens
- Candidate profiles are embedded from a constructed skill sentence
- Synthetic candidate text fields carry no usable semantic signal
- Skill gap operates on canonical tokens, not ESCO URIs
- No similarity thresholds are calibrated in this notebook

## Inputs
- outputs/normalized_candidate_skill_profiles.csv
- outputs/curated_job_descriptions.csv
- outputs/ats_scored_candidates.csv
- outputs/ats_scoring_artifacts.json

## Outputs
- outputs/semantic_match_scores.csv
- outputs/skill_gap_outputs.csv
- outputs/embedding_artifacts.pkl

In [1]:
import pandas as pd
import numpy as np
import json
import pickle
from pathlib import Path

OUTPUT_DIR = Path("../outputs")

# loading input artifacts
profiles   = pd.read_csv(OUTPUT_DIR / "normalized_candidate_skill_profiles.csv")
job_desc   = pd.read_csv(OUTPUT_DIR / "curated_job_descriptions.csv")
ats_scores = pd.read_csv(OUTPUT_DIR / "ats_scored_candidates.csv")

with open(OUTPUT_DIR / "ats_scoring_artifacts.json") as f:
    scoring_artifacts = json.load(f)

# validating shapes
print("profiles shape:   ", profiles.shape)
print("job_desc shape:   ", job_desc.shape)
print("ats_scores shape: ", ats_scores.shape)
print()

# validating required columns
required_profiles = ["candidate_id", "normalized_skill_profile", "skill_count"]
required_jobs     = ["job_id", "title", "description", "domain",
                     "normalized_skills", "skill_count"]
required_ats      = ["candidate_id", "primary_domain", "ats_score"]

for col in required_profiles:
    assert col in profiles.columns, f"Missing in profiles: {col}"
for col in required_jobs:
    assert col in job_desc.columns, f"Missing in job_desc: {col}"
for col in required_ats:
    assert col in ats_scores.columns, f"Missing in ats_scores: {col}"

print("All required columns present.")
print()

# validating domain coverage
print("Job description domain distribution:")
print(job_desc["domain"].value_counts().to_string())
print()

# checking for nulls in critical fields
print("Null normalized_skill_profile: ",
      profiles["normalized_skill_profile"].isnull().sum())
print("Null job description text:     ",
      job_desc["description"].isnull().sum())
print("Null normalized_skills (jobs): ",
      job_desc["normalized_skills"].isnull().sum())
print()

# merging domain onto profiles — profiles do not carry domain directly
profiles = profiles.merge(
    ats_scores[["candidate_id", "primary_domain"]],
    on="candidate_id",
    how="left"
)

print("Profiles with domain assigned: ",
      profiles["primary_domain"].notna().sum(), "/", len(profiles))
print()
print("Candidate domain distribution:")
print(profiles["primary_domain"].value_counts().to_string())

profiles shape:    (5000, 5)
job_desc shape:    (252, 13)
ats_scores shape:  (5000, 13)

All required columns present.

Job description domain distribution:
domain
IT              42
Data Science    42
HR              42
Legal           42
Engineering     42
Management      42

Null normalized_skill_profile:  0
Null job description text:      0
Null normalized_skills (jobs):  18

Profiles with domain assigned:  5000 / 5000

Candidate domain distribution:
primary_domain
IT              2772
Data Science    1002
HR               501
Legal            397
Engineering      189
Management       139


## Section 1: Candidate Skill Sentence Construction

Each candidate's normalized skill profile is a pipe-delimited string of canonical tokens.
To embed these profiles meaningfully, the tokens are joined into a natural language sentence
of the form "Skills include: token1, token2, token3".

This phrasing is chosen deliberately. Sentence-transformer models are trained on natural
language pairs. A grammatical sentence produces better token contextualization than a
bare comma-separated list. The improvement is modest at this profile length but costs nothing.

The constructed sentence is the sole input to the candidate embedding step.
Synthetic text fields (experience_summary, project_summary) are excluded — they carry
no semantic variance across candidates and would compress all embeddings toward a single point.

In [2]:
# constructing natural language skill sentences from normalized profiles
def build_skill_sentence(pipe_delimited_profile):
    if pd.isna(pipe_delimited_profile) or pipe_delimited_profile.strip() == "":
        return "Skills include: not specified"
    tokens = [t.strip() for t in pipe_delimited_profile.split("|") if t.strip()]
    return "Skills include: " + ", ".join(tokens)

profiles["skill_sentence"] = profiles["normalized_skill_profile"].apply(
    build_skill_sentence
)

# spot checking sentence construction
print("Sample skill sentences:")
print()
for _, row in profiles.sample(5, random_state=42).iterrows():
    print(f"  [{row['primary_domain']}] {row['candidate_id']}")
    print(f"  Profile:  {row['normalized_skill_profile']}")
    print(f"  Sentence: {row['skill_sentence']}")
    print()

# confirming no empty sentences
empty_sentences = (profiles["skill_sentence"].str.strip() == "").sum()
print(f"Empty skill sentences: {empty_sentences}")
print(f"Total skill sentences: {len(profiles)}")
print()

# checking sentence length distribution
sentence_lengths = profiles["skill_sentence"].str.len()
print("Skill sentence character length — min/mean/max:",
      sentence_lengths.min(),
      round(sentence_lengths.mean(), 1),
      sentence_lengths.max())

Sample skill sentences:

  [HR] CND_101502
  Profile:  employee engagement|gcp|hr analytics|manage payroll|recruit employees|use online tools to collaborate
  Sentence: Skills include: employee engagement, gcp, hr analytics, manage payroll, recruit employees, use online tools to collaborate

  [HR] CND_102587
  Profile:  employee engagement|hr analytics|hrms|jira|recruit employees|servicenow
  Sentence: Skills include: employee engagement, hr analytics, hrms, jira, recruit employees, servicenow

  [IT] CND_102654
  Profile:  azure|docker|java (computer programming)|jira|linux|sql
  Sentence: Skills include: azure, docker, java (computer programming), jira, linux, sql

  [Data Science] CND_101056
  Profile:  jira|natural language processing|power bi|python (computer programming)|servicenow|tensorflow
  Sentence: Skills include: jira, natural language processing, power bi, python (computer programming), servicenow, tensorflow

  [IT] CND_100706
  Profile:  aws|docker|gcp|java (computer p

## Section 2: Embedding Model and Job Description Embeddings

All embeddings are computed using sentence-transformers with all-MiniLM-L6-v2.
This model produces 384-dimensional vectors, runs on CPU within acceptable latency,
and fits within Streamlit Cloud memory constraints when loaded with st.cache_resource.

Job descriptions are embedded from their full description text, not from extracted
skill tokens. This uses the richer signal available in the posting and is the correct
production approach. The 18 postings with zero extracted skills still have substantive
description text and will produce meaningful embeddings.

Job description embeddings are computed first and persisted. They are a fixed artifact
that does not change between notebook runs or application sessions.

Embeddings are computed in batches to manage memory. Batch size 32 is appropriate
for this hardware configuration.

In [4]:
# installing sentence-transformers if not available
import importlib

from sentence_transformers import SentenceTransformer
import time

# loading model
print("Loading sentence-transformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")
print()

# embedding job descriptions from full description text
print(f"Embedding {len(job_desc)} job descriptions...")
start = time.time()

job_embeddings = model.encode(
    job_desc["description"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

elapsed = round(time.time() - start, 1)
print()
print(f"Job description embedding complete: {elapsed}s")
print(f"Embedding matrix shape: {job_embeddings.shape}")
print(f"Embedding dtype:        {job_embeddings.dtype}")
print()

# building job index mapping — job_id to embedding row index
job_id_to_idx = {
    job_id: idx
    for idx, job_id in enumerate(job_desc["job_id"].tolist())
}

# spot checking embedding norms — should all be 1.0 with normalize_embeddings=True
sample_norms = np.linalg.norm(job_embeddings[:5], axis=1)
print("Sample embedding L2 norms (expect 1.0):")
print(sample_norms.round(6))

Loading sentence-transformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6994.24it/s]


Model loaded.

Embedding 252 job descriptions...


Batches: 100%|██████████| 8/8 [00:08<00:00,  1.08s/it]


Job description embedding complete: 8.7s
Embedding matrix shape: (252, 384)
Embedding dtype:        float32

Sample embedding L2 norms (expect 1.0):
[1. 1. 1. 1. 1.]


## Section 3: Candidate Skill Sentence Embeddings

Candidate profiles are embedded from the constructed skill sentences produced in Section 1.
The same model and configuration are used for both job descriptions and candidate profiles
to ensure the embedding space is shared and cosine similarity is meaningful.

5,000 embeddings at batch size 32 requires approximately 157 forward passes.
Expected runtime on this hardware is 60-120 seconds.

A candidate_id to embedding index mapping is built alongside the matrix
for deterministic lookup during similarity computation and Streamlit application use.

In [5]:
# embedding candidate skill sentences
print(f"Embedding {len(profiles)} candidate skill sentences...")
start = time.time()

candidate_embeddings = model.encode(
    profiles["skill_sentence"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

elapsed = round(time.time() - start, 1)
print()
print(f"Candidate embedding complete: {elapsed}s")
print(f"Embedding matrix shape: {candidate_embeddings.shape}")
print(f"Embedding dtype:        {candidate_embeddings.dtype}")
print()

# building candidate index mapping
candidate_id_to_idx = {
    cid: idx
    for idx, cid in enumerate(profiles["candidate_id"].tolist())
}

# spot checking norms
sample_norms = np.linalg.norm(candidate_embeddings[:5], axis=1)
print("Sample candidate embedding L2 norms (expect 1.0):")
print(sample_norms.round(6))
print()

# checking embedding variance across candidates
# with synthetic skill profiles, variance may be compressed
# this is a diagnostic, not a pass/fail check
embedding_std = candidate_embeddings.std(axis=0)
print(f"Mean per-dimension std across all candidates: {embedding_std.mean():.4f}")
print(f"Min per-dimension std:                        {embedding_std.min():.4f}")
print(f"Max per-dimension std:                        {embedding_std.max():.4f}")
print()

# checking inter-domain separation as a sanity check
# candidates from different domains should have lower mean similarity
# than candidates within the same domain
domain_labels = profiles["primary_domain"].tolist()
sample_it  = candidate_embeddings[profiles["primary_domain"] == "IT"][:50]
sample_ds  = candidate_embeddings[profiles["primary_domain"] == "Data Science"][:50]
sample_hr  = candidate_embeddings[profiles["primary_domain"] == "HR"][:50]

mean_it_it = (sample_it @ sample_it.T).mean()
mean_it_ds = (sample_it @ sample_ds.T).mean()
mean_it_hr = (sample_it @ sample_hr.T).mean()

print("Mean cosine similarity — IT vs IT:           ", round(float(mean_it_it), 4))
print("Mean cosine similarity — IT vs Data Science: ", round(float(mean_it_ds), 4))
print("Mean cosine similarity — IT vs HR:           ", round(float(mean_it_hr), 4))

Embedding 5000 candidate skill sentences...


Batches: 100%|██████████| 157/157 [00:21<00:00,  7.46it/s]


Candidate embedding complete: 21.1s
Embedding matrix shape: (5000, 384)
Embedding dtype:        float32

Sample candidate embedding L2 norms (expect 1.0):
[1. 1. 1. 1. 1.]

Mean per-dimension std across all candidates: 0.0254
Min per-dimension std:                        0.0000
Max per-dimension std:                        0.0476

Mean cosine similarity — IT vs IT:            0.8361
Mean cosine similarity — IT vs Data Science:  0.7508
Mean cosine similarity — IT vs HR:            0.6769


## Section 4: Semantic Similarity Scoring

Cosine similarity between candidate embeddings and job description embeddings is computed
as a dot product since both matrices are L2-normalized.

Each candidate is scored only against job descriptions in their own domain.
Cross-domain scoring is not computed here — it belongs to the Streamlit application
where a user selects a specific job description regardless of domain.

Per-candidate outputs:
- top_job_similarity: highest cosine similarity against any job in the candidate's domain
- mean_domain_similarity: mean cosine similarity across all jobs in the candidate's domain
- top_job_id: job_id of the best-matching job description
- display_score: top_job_similarity rescaled linearly to 0-100

Rescaling uses the observed population min and max rather than theoretical bounds.
No thresholds are calibrated. Raw cosine scores are retained alongside display scores.

Synthetic data limitation: candidate skill sentences are short (5-6 tokens) while
job descriptions are full text. This asymmetry compresses similarity scores toward
the upper range. Score distributions on real resume inputs will differ.

In [6]:
# building domain-stratified job index
# mapping each domain to the row indices of its job descriptions in the embedding matrix
domain_job_indices = {}
for domain in job_desc["domain"].unique():
    indices = job_desc[job_desc["domain"] == domain].index.tolist()
    domain_job_indices[domain] = indices

print("Job indices per domain:")
for domain, indices in sorted(domain_job_indices.items()):
    print(f"  {domain:<15} {len(indices)} jobs, "
          f"index range {min(indices)}-{max(indices)}")
print()

# computing per-candidate similarity scores against domain job pool
results = []

for domain in sorted(profiles["primary_domain"].unique()):
    domain_mask        = profiles["primary_domain"] == domain
    domain_candidates  = profiles[domain_mask]
    domain_cand_embs   = candidate_embeddings[domain_mask.values]
    domain_job_idx     = domain_job_indices[domain]
    domain_job_embs    = job_embeddings[domain_job_idx]

    # dot product is cosine similarity since embeddings are normalized
    # shape: (n_candidates_in_domain, n_jobs_in_domain)
    sim_matrix = domain_cand_embs @ domain_job_embs.T

    top_sim      = sim_matrix.max(axis=1)
    mean_sim     = sim_matrix.mean(axis=1)
    top_job_pos  = sim_matrix.argmax(axis=1)

    # mapping top job position back to actual job_id
    top_job_ids = [
        job_desc.iloc[domain_job_idx[pos]]["job_id"]
        for pos in top_job_pos
    ]

    for i, (_, row) in enumerate(domain_candidates.iterrows()):
        results.append({
            "candidate_id":          row["candidate_id"],
            "primary_domain":        domain,
            "top_job_similarity":    round(float(top_sim[i]), 6),
            "mean_domain_similarity": round(float(mean_sim[i]), 6),
            "top_job_id":            top_job_ids[i]
        })

similarity_df = pd.DataFrame(results)

print(f"Similarity scores computed: {len(similarity_df)} candidates")
print()
print("top_job_similarity distribution:")
print(similarity_df["top_job_similarity"].describe().round(4).to_string())
print()
print("Mean top_job_similarity by domain:")
print(similarity_df.groupby("primary_domain")["top_job_similarity"]
      .mean().round(4).sort_values(ascending=False).to_string())

Job indices per domain:
  Data Science    42 jobs, index range 42-83
  Engineering     42 jobs, index range 168-209
  HR              42 jobs, index range 84-125
  IT              42 jobs, index range 0-41
  Legal           42 jobs, index range 126-167
  Management      42 jobs, index range 210-251

Similarity scores computed: 5000 candidates

top_job_similarity distribution:
count    5000.0000
mean        0.5414
std         0.0482
min         0.4104
25%         0.5091
50%         0.5428
75%         0.5720
max         0.6868

Mean top_job_similarity by domain:
primary_domain
Legal           0.6172
IT              0.5539
Engineering     0.5453
HR              0.5300
Data Science    0.4957
Management      0.4424


## Section 4 (continued): Display Score Rescaling

Raw cosine similarity scores range from 0.41 to 0.69 in this population.
Presenting these values directly in the Streamlit application would be unintuitive.
A score of 0.54 carries no meaning to a non-technical user.

Scores are rescaled linearly to a 0-100 display range using the observed
population min and max. This preserves relative rank ordering and within-domain
differentiation while producing interpretable output.

The rescaling parameters are stored in the scoring artifacts for consistent
application to new candidates at runtime. A new candidate whose raw similarity
falls outside the population range is clipped to 0 or 100 rather than extrapolating.

This rescaling is a display transformation only. All downstream computations
in Notebook 07 use raw cosine similarity scores.

In [7]:
# computing display score using population min/max rescaling
pop_min = similarity_df["top_job_similarity"].min()
pop_max = similarity_df["top_job_similarity"].max()

print(f"Population similarity range: {pop_min:.4f} to {pop_max:.4f}")
print()

def rescale_to_display(raw_score, pop_min, pop_max):
    if pop_max == pop_min:
        return 50.0
    scaled = (raw_score - pop_min) / (pop_max - pop_min) * 100
    return round(float(np.clip(scaled, 0, 100)), 2)

similarity_df["display_score"] = similarity_df["top_job_similarity"].apply(
    lambda x: rescale_to_display(x, pop_min, pop_max)
)

print("display_score distribution:")
print(similarity_df["display_score"].describe().round(2).to_string())
print()

print("Mean display_score by domain:")
print(similarity_df.groupby("primary_domain")["display_score"]
      .mean().round(2).sort_values(ascending=False).to_string())
print()

# spot checking rescaling correctness
print("Rescaling spot checks:")
print(f"  min raw {pop_min:.4f} -> display: "
      f"{rescale_to_display(pop_min, pop_min, pop_max)}")
print(f"  max raw {pop_max:.4f} -> display: "
      f"{rescale_to_display(pop_max, pop_min, pop_max)}")
print(f"  mid raw {(pop_min+pop_max)/2:.4f} -> display: "
      f"{rescale_to_display((pop_min+pop_max)/2, pop_min, pop_max)}")
print()

# storing rescaling parameters for Streamlit application use
similarity_rescaling = {
    "pop_min": round(float(pop_min), 6),
    "pop_max": round(float(pop_max), 6),
    "method":  "linear_minmax",
    "clip":    True,
    "note":    "Derived from 5000 synthetic candidates. Recalibrate on real resume population."
}

print("Rescaling parameters:")
for k, v in similarity_rescaling.items():
    print(f"  {k}: {v}")

Population similarity range: 0.4104 to 0.6868

display_score distribution:
count    5000.00
mean       47.41
std        17.45
min         0.00
25%        35.72
50%        47.89
75%        58.49
max       100.00

Mean display_score by domain:
primary_domain
Legal           74.82
IT              51.92
Engineering     48.80
HR              43.27
Data Science    30.85
Management      11.58

Rescaling spot checks:
  min raw 0.4104 -> display: 0.0
  max raw 0.6868 -> display: 100.0
  mid raw 0.5486 -> display: 50.0

Rescaling parameters:
  pop_min: 0.410421
  pop_max: 0.686757
  method: linear_minmax
  clip: True
  note: Derived from 5000 synthetic candidates. Recalibrate on real resume population.


## Section 5: Skill Gap Analysis

Skill gap is computed as a set difference between job description required skills
and candidate skills, both represented as canonical token sets.

For each candidate, the comparison target is the union of canonical skills extracted
from all job descriptions in their domain. This produces a domain-level gap signal
suitable for population benchmarking.

Missing skills are ranked by their frequency of appearance across domain job descriptions.
A skill that appears in 30 of 42 domain job postings is a more critical gap than one
appearing in 3. Frequency rank reflects hiring market demand within the domain.

The Streamlit application will compute gap against a single selected job description
at runtime. The domain-aggregate gap computed here is for benchmarking only.

Known limitation: mean extracted skills per job description is 2.81, lower than
candidate profiles at 5.91. Gap analysis reflects vocabulary-limited extraction
from job descriptions, not true skill requirements. Candidates may appear to have
strong coverage simply because few skills were extracted from postings.
This limitation applies to all gap outputs from this notebook.

In [8]:
# building domain-level skill frequency from job descriptions
# counting how many postings in each domain contain each canonical skill
domain_skill_freq = {}

for domain in job_desc["domain"].unique():
    domain_jobs = job_desc[job_desc["domain"] == domain]
    skill_counts = {}

    for skills_str in domain_jobs["normalized_skills"]:
        if pd.isna(skills_str) or skills_str.strip() == "":
            continue
        for skill in skills_str.split("|"):
            skill = skill.strip()
            if skill:
                skill_counts[skill] = skill_counts.get(skill, 0) + 1

    domain_skill_freq[domain] = skill_counts

# inspecting domain skill frequency to validate extraction quality
print("Domain skill frequency summary:")
print()
for domain in sorted(domain_skill_freq.keys()):
    freq = domain_skill_freq[domain]
    total_unique = len(freq)
    top_5 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  {domain} — {total_unique} unique skills in job pool")
    for skill, count in top_5:
        print(f"    {skill:<50} {count}/42 postings")
    print()

Domain skill frequency summary:

  Data Science — 24 unique skills in job pool
    machine learning                                   29/42 postings
    sql                                                28/42 postings
    python (computer programming)                      24/42 postings
    strategy                                           17/42 postings
    ensure ongoing compliance with regulations         17/42 postings

  Engineering — 19 unique skills in job pool
    manufacturing                                      23/42 postings
    ensure ongoing compliance with regulations         16/42 postings
    strategy                                           15/42 postings
    cad                                                8/42 postings
    recruit employees                                  7/42 postings

  HR — 12 unique skills in job pool
    ensure ongoing compliance with regulations         23/42 postings
    strategy                                           18/42 postings


## Section 5 (continued): Domain Skill Frequency Filtering

Inspection of domain skill frequencies reveals two classes of noise in job description
skill extraction:

1. Cross-domain ubiquitous tokens: "strategy" and "ensure ongoing compliance with
   regulations" appear as top skills in all six domains. These tokens appear in job
   description prose as general business language rather than as specific skill
   requirements. They suppress genuine domain signal in gap analysis.

2. Domain-implausible tokens: "recruit employees" appearing in Engineering and Legal
   postings reflects incidental text matches, not genuine skill requirements.

A domain-relevance filter is applied to the frequency table before gap analysis.
A skill is considered domain-relevant if it meets both conditions:
- It appears in fewer than 5 of 6 domains with non-zero frequency, OR
- It is the highest-frequency skill in at least one domain

This preserves genuine cross-domain skills (aws, sql, python) while suppressing
tokens that are distributed uniformly across all domains purely as language artifacts.
The filter is applied to the frequency table only. Raw extraction artifacts are unchanged.

In [9]:
# identifying tokens present across too many domains to carry domain signal
skill_domain_presence = {}

for domain, freq in domain_skill_freq.items():
    for skill in freq:
        skill_domain_presence[skill] = skill_domain_presence.get(skill, 0) + 1

print("Skills present in multiple domains:")
print()
for skill, n_domains in sorted(skill_domain_presence.items(),
                                key=lambda x: x[1], reverse=True):
    if n_domains >= 4:
        print(f"  {skill:<55} present in {n_domains}/6 domains")
print()

# defining cross-domain noise tokens to suppress in gap analysis
# these appear uniformly across domains as language artifacts, not skill signals
GAP_SUPPRESS_TOKENS = {
    "ensure ongoing compliance with regulations",
    "strategy"
}

print("Tokens suppressed from gap analysis:")
for t in sorted(GAP_SUPPRESS_TOKENS):
    print(f"  {t}")
print()

# rebuilding filtered domain skill frequency tables
domain_skill_freq_filtered = {}

for domain, freq in domain_skill_freq.items():
    filtered = {
        skill: count
        for skill, count in freq.items()
        if skill not in GAP_SUPPRESS_TOKENS
    }
    domain_skill_freq_filtered[domain] = filtered

# confirming filter effect
print("Domain skill pools after filtering:")
print()
for domain in sorted(domain_skill_freq_filtered.keys()):
    freq = domain_skill_freq_filtered[domain]
    top_5 = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"  {domain} — {len(freq)} unique skills remaining")
    for skill, count in top_5:
        print(f"    {skill:<50} {count}/42 postings")
    print()

Skills present in multiple domains:

  ensure ongoing compliance with regulations              present in 6/6 domains
  strategy                                                present in 6/6 domains
  risk management                                         present in 6/6 domains
  recruit employees                                       present in 6/6 domains
  manufacturing                                           present in 6/6 domains
  aws                                                     present in 5/6 domains
  ict project management methodologies                    present in 5/6 domains
  java (computer programming)                             present in 5/6 domains
  sql                                                     present in 5/6 domains
  azure                                                   present in 4/6 domains
  python (computer programming)                           present in 4/6 domains
  due diligence                                           present in 4/6

## Section 5 (continued): Skill Gap Computation

For each candidate, missing skills are identified as canonical tokens present in their
domain's filtered job pool but absent from their normalized skill profile.

Missing skills are ranked by domain posting frequency — skills appearing in more
postings within the domain are ranked as higher priority gaps.

Three outputs are computed per candidate:
- missing_skills: pipe-delimited list of missing canonical tokens, ranked by domain frequency
- gap_count: number of missing skills
- domain_coverage_pct: percentage of domain skill pool tokens the candidate already holds

Domain coverage is computed against the filtered pool only.
Suppressed tokens (strategy, ensure ongoing compliance) are excluded from both
the numerator and denominator of coverage calculation.

The 5,000-record gap outputs are for population benchmarking in Notebook 07.
The Streamlit application computes gap against a single selected job description at runtime.

In [10]:
# computing skill gap for all 5000 candidates
gap_results = []

for _, row in profiles.iterrows():
    domain   = row["primary_domain"]
    cand_id  = row["candidate_id"]

    # parsing candidate normalized skill profile into a set
    profile_str = row["normalized_skill_profile"]
    if pd.isna(profile_str) or profile_str.strip() == "":
        candidate_skills = set()
    else:
        candidate_skills = set(
            s.strip() for s in profile_str.split("|") if s.strip()
        )

    # retrieving filtered domain skill pool with frequencies
    domain_freq = domain_skill_freq_filtered.get(domain, {})

    # identifying missing skills — in domain pool but not in candidate profile
    missing = {
        skill: count
        for skill, count in domain_freq.items()
        if skill not in candidate_skills
    }

    # ranking missing skills by domain posting frequency descending
    ranked_missing = sorted(missing.items(), key=lambda x: x[1], reverse=True)
    missing_skill_names = [s for s, _ in ranked_missing]

    # computing domain coverage
    domain_pool_size = len(domain_freq)
    matched = len(candidate_skills & set(domain_freq.keys()))
    coverage_pct = round(matched / domain_pool_size * 100, 2) if domain_pool_size > 0 else 0.0

    gap_results.append({
        "candidate_id":        cand_id,
        "primary_domain":      domain,
        "missing_skills":      "|".join(missing_skill_names),
        "gap_count":           len(missing_skill_names),
        "domain_coverage_pct": coverage_pct
    })

gap_df = pd.DataFrame(gap_results)

print(f"Gap analysis complete: {len(gap_df)} candidates")
print()
print("gap_count distribution:")
print(gap_df["gap_count"].describe().round(2).to_string())
print()
print("domain_coverage_pct distribution:")
print(gap_df["domain_coverage_pct"].describe().round(2).to_string())
print()
print("Mean coverage and gap by domain:")
print(gap_df.groupby("primary_domain")[["domain_coverage_pct", "gap_count"]]
      .mean().round(2).sort_values("domain_coverage_pct", ascending=False)
      .to_string())
print()

# spot checking a sample candidate gap output
sample = gap_df[gap_df["primary_domain"] == "Data Science"].iloc[0]
print("Sample gap output — Data Science candidate:")
print(f"  candidate_id:        {sample['candidate_id']}")
print(f"  missing_skills:      {sample['missing_skills']}")
print(f"  gap_count:           {sample['gap_count']}")
print(f"  domain_coverage_pct: {sample['domain_coverage_pct']}")

Gap analysis complete: 5000 candidates

gap_count distribution:
count    5000.00
mean       12.81
std         2.86
min         6.00
25%        12.00
50%        13.00
75%        14.00
max        18.00

domain_coverage_pct distribution:
count    5000.00
mean       26.83
std         4.25
min        16.67
25%        22.73
50%        27.78
75%        27.78
max        40.00

Mean coverage and gap by domain:
                domain_coverage_pct  gap_count
primary_domain                                
HR                            31.86       6.81
Engineering                   31.53      11.64
IT                            26.93      13.15
Legal                         25.15       8.98
Data Science                  24.36      16.64
Management                    23.02      12.32

Sample gap output — Data Science candidate:
  candidate_id:        CND_100001
  missing_skills:      sql|azure|ict project management methodologies|natural language processing|gcp|docker|recruit employees|risk manageme

## Section 6: Artifact Export

Three artifacts are exported:

**embedding_artifacts.pkl**
Contains job description embeddings, candidate skill sentence embeddings,
index mappings for both, domain-to-job-index mapping, and similarity rescaling
parameters. This file is the complete reusable embedding state for the
Streamlit application. It must be loaded once at application startup and
cached with st.cache_resource.

**semantic_match_scores.csv**
Per-candidate similarity scores including raw cosine similarity, mean domain
similarity, best-matching job_id, and display score. Used by Notebook 07
for benchmarking and by the Streamlit application for score display.

**skill_gap_outputs.csv**
Per-candidate skill gap outputs including missing skills ranked by domain
frequency, gap count, and domain coverage percentage. Used by Notebook 07
and the Streamlit application for gap display and recommendation generation.

All artifacts use candidate_id as the join key consistent with upstream notebooks.

In [11]:
# assembling embedding artifacts bundle
embedding_artifacts = {
    # embedding matrices
    "job_embeddings":             job_embeddings,
    "candidate_embeddings":       candidate_embeddings,

    # index mappings
    "job_id_to_idx":              job_id_to_idx,
    "candidate_id_to_idx":        candidate_id_to_idx,

    # domain to job index mapping for domain-stratified lookup
    "domain_job_indices":         domain_job_indices,

    # similarity rescaling parameters for Streamlit display
    "similarity_rescaling":       similarity_rescaling,

    # domain skill frequency tables for runtime gap analysis
    "domain_skill_freq_filtered": domain_skill_freq_filtered,

    # suppressed tokens — needed for consistent gap computation at runtime
    "gap_suppress_tokens":        list(GAP_SUPPRESS_TOKENS),

    # metadata
    "model_name":                 "all-MiniLM-L6-v2",
    "embedding_dim":              384,
    "n_job_descriptions":         len(job_desc),
    "n_candidates":               len(profiles),
    "normalize_embeddings":       True
}

# exporting embedding artifacts
artifacts_path = OUTPUT_DIR / "embedding_artifacts.pkl"
with open(artifacts_path, "wb") as f:
    pickle.dump(embedding_artifacts, f, protocol=4)

artifacts_size_mb = round(artifacts_path.stat().st_size / (1024 * 1024), 2)
print(f"embedding_artifacts.pkl exported: {artifacts_size_mb} MB")
print()

# exporting similarity scores
similarity_df.to_csv(OUTPUT_DIR / "semantic_match_scores.csv", index=False)
sim_size_kb = round(
    (OUTPUT_DIR / "semantic_match_scores.csv").stat().st_size / 1024, 1
)
print(f"semantic_match_scores.csv exported: {sim_size_kb} KB  "
      f"({len(similarity_df)} rows, {len(similarity_df.columns)} columns)")
print()

# exporting skill gap outputs
gap_df.to_csv(OUTPUT_DIR / "skill_gap_outputs.csv", index=False)
gap_size_kb = round(
    (OUTPUT_DIR / "skill_gap_outputs.csv").stat().st_size / 1024, 1
)
print(f"skill_gap_outputs.csv exported: {gap_size_kb} KB  "
      f"({len(gap_df)} rows, {len(gap_df.columns)} columns)")
print()

# final validation — reloading all three artifacts
print("Reloading and validating exported artifacts...")
print()

with open(OUTPUT_DIR / "embedding_artifacts.pkl", "rb") as f:
    reloaded_artifacts = pickle.load(f)

sim_check = pd.read_csv(OUTPUT_DIR / "semantic_match_scores.csv")
gap_check = pd.read_csv(OUTPUT_DIR / "skill_gap_outputs.csv")

print("embedding_artifacts.pkl keys:")
for k, v in reloaded_artifacts.items():
    if isinstance(v, np.ndarray):
        print(f"  {k:<35} ndarray {v.shape} {v.dtype}")
    elif isinstance(v, dict):
        print(f"  {k:<35} dict    {len(v)} entries")
    elif isinstance(v, list):
        print(f"  {k:<35} list    {len(v)} items")
    else:
        print(f"  {k:<35} {type(v).__name__}  {v}")
print()

print(f"semantic_match_scores.csv:  {sim_check.shape}  "
      f"nulls: {sim_check.isnull().sum().sum()}")
print(f"skill_gap_outputs.csv:      {gap_check.shape}  "
      f"nulls: {gap_check.isnull().sum().sum()}")
print()

# confirming all output files present
print("outputs/ directory contents:")
for f in sorted(OUTPUT_DIR.glob("*")):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name:<55} {size_kb} KB")

embedding_artifacts.pkl exported: 7.77 MB

semantic_match_scores.csv exported: 256.3 KB  (5000 rows, 6 columns)

skill_gap_outputs.csv exported: 1124.4 KB  (5000 rows, 5 columns)

Reloading and validating exported artifacts...

embedding_artifacts.pkl keys:
  job_embeddings                      ndarray (252, 384) float32
  candidate_embeddings                ndarray (5000, 384) float32
  job_id_to_idx                       dict    252 entries
  candidate_id_to_idx                 dict    5000 entries
  domain_job_indices                  dict    6 entries
  similarity_rescaling                dict    5 entries
  domain_skill_freq_filtered          dict    6 entries
  gap_suppress_tokens                 list    2 items
  model_name                          str  all-MiniLM-L6-v2
  embedding_dim                       int  384
  n_job_descriptions                  int  252
  n_candidates                        int  5000
  normalize_embeddings                bool  True

semantic_match_score

# Notebook 06: Summary

## Status
Complete. All artifacts exported and validated.

## Artifacts Produced

| Artifact | Rows | Size |
|---|---|---|
| outputs/semantic_match_scores.csv | 5000 x 6 | 256 KB |
| outputs/skill_gap_outputs.csv | 5000 x 5 | 1.1 MB |
| outputs/embedding_artifacts.pkl | — | 7.77 MB |

## Methodology

Job descriptions were embedded from full description text using all-MiniLM-L6-v2
with L2 normalization. Candidate profiles were embedded from a constructed skill
sentence of the form "Skills include: token1, token2" using the same model.
Cosine similarity was computed as a dot product on normalized embeddings.
Scoring was domain-stratified — each candidate scored against their own domain's
42 job descriptions only.

Skill gap was computed as a set difference between the candidate's normalized skill
profile and the filtered domain skill frequency table, with missing skills ranked
by domain posting frequency.

## Key Findings

**Embedding quality.** Cross-domain separation is working correctly.
IT vs IT mean similarity 0.836, IT vs Data Science 0.751, IT vs HR 0.677.
Domain ordering is preserved and interpretable.

**Score compression.** Raw cosine similarity ranges from 0.41 to 0.69 with
mean 0.54 and std 0.048. Compression is caused by the asymmetry between
short skill sentences (5-6 tokens) and full job description text (mean 9,392
characters). This is a synthetic data artifact.

**Display score rescaling.** Linear min-max rescaling maps population bounds
to 0-100. Domain display score means: Legal 74.82, IT 51.92, Engineering 48.80,
HR 43.27, Data Science 30.85, Management 11.58. Wide domain spread reflects
vocabulary alignment differences, not candidate quality differences.
Recalibration required on real resume data.

**Gap analysis.** Mean gap count 12.81, mean domain coverage 26.83 percent.
Both figures are inflated by the mismatch between 5-6 token candidate profiles
and 10-22 token domain skill pools. Two tokens suppressed from gap analysis —
"strategy" and "ensure ongoing compliance with regulations" — due to uniform
cross-domain appearance as language artifacts rather than skill signals.

## Decisions Made

- Embedding model: all-MiniLM-L6-v2, normalize_embeddings=True
- Job descriptions embedded from full text
- Candidates embedded from constructed skill sentence
- Synthetic candidate text fields excluded entirely
- Similarity scoring domain-stratified only
- Display score: linear min-max rescaling, parameters stored in pkl
- Gap suppress tokens: strategy, ensure ongoing compliance with regulations
- Missing skills ranked by domain posting frequency descending
- Storage: CSV for tabular artifacts, PKL for embedding matrices and metadata

## Known Limitations

All similarity scores and gap metrics are derived from synthetic data and are
not production-calibrated. Display score rescaling parameters, gap frequency
tables, and suppression lists must be recalibrated once real resume data is
available. Absolute gap counts and coverage percentages should not be used
as hard thresholds in the Streamlit application without recalibration.

## Decisions Deferred to Notebook 07

Benchmarking methodology approved but not implemented. Domain-stratified
percentile ranks will be computed for ATS score, semantic match display score,
and domain coverage percentage. Low confidence flag applied to Management
(139 records) and Engineering (189 records). Experience tail candidates
(11-20 years) binned into a single senior bucket. No crossed domain-experience
percentiles. Full architecture recorded in Memory V7.